# Day 5: Comprehensive Model Evaluation

Detailed evaluation of the trained Linear Regression model with multiple metrics and visualizations.

## 📋 Objectives
- Calculate comprehensive evaluation metrics
- Compare train vs test performance
- Analyze prediction errors
- Generate evaluation report

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import json

from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    mean_absolute_percentage_error, median_absolute_error,
    max_error, explained_variance_score
)
from sklearn.model_selection import learning_curve, validation_curve

# Styling
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_context('notebook', font_scale=1.1)

print("✅ Libraries imported")

In [ ]:
# Load model, data, and metadata
model = joblib.load('../../day12/models/linear_regression_model.pkl')
scaler = joblib.load('../../day12/models/scaler.pkl')

X = pd.read_csv('../../day12/data/features_scaled.csv')
y = pd.read_csv('../../day12/data/target.csv').squeeze()

with open('../../day12/models/model_metadata.json', 'r') as f:
    metadata = json.load(f)

print("✅ Model and data loaded")
print(f"Features: {X.shape[1]}, Samples: {X.shape[0]}")

In [ ]:
# Split data (same as training)
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, shuffle=True
)

# Predictions
y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)

# Comprehensive Metrics Function
def calculate_metrics(y_true, y_pred, prefix=""):
    """Calculate all regression metrics."""
    metrics = {
        f'{prefix}MAE': mean_absolute_error(y_true, y_pred),
        f'{prefix}MSE': mean_squared_error(y_true, y_pred),
        f'{prefix}RMSE': np.sqrt(mean_squared_error(y_true, y_pred)),
        f'{prefix}R²': r2_score(y_true, y_pred),
        f'{prefix}MAPE': mean_absolute_percentage_error(y_true, y_pred) * 100,
        f'{prefix}MedAE': median_absolute_error(y_true, y_pred),
        f'{prefix}MaxError': max_error(y_true, y_pred),
        f'{prefix}ExplainedVar': explained_variance_score(y_true, y_pred)
    }
    return metrics

train_metrics = calculate_metrics(y_train, y_train_pred, "Train_")
test_metrics = calculate_metrics(y_test, y_test_pred, "Test_")

# Display metrics table
metrics_df = pd.DataFrame({
    'Metric': list(train_metrics.keys()),
    'Train': list(train_metrics.values()),
    'Test': list(test_metrics.values())
})
metrics_df['Difference'] = metrics_df['Test'] - metrics_df['Train']

print("📊 COMPREHENSIVE EVALUATION METRICS")
print("=" * 60)
display(metrics_df.style.format({'Train': '{:.4f}', 'Test': '{:.4f}', 'Difference': '{:.4f}'}))

In [ ]:
# Learning Curves
train_sizes, train_scores, test_scores = learning_curve(
    model, X, y, cv=5, scoring='r2',
    train_sizes=np.linspace(0.1, 1.0, 10),
    random_state=42, n_jobs=-1
)

train_mean = np.mean(train_scores, axis=1)
train_std = np.std(train_scores, axis=1)
test_mean = np.mean(test_scores, axis=1)
test_std = np.std(test_scores, axis=1)

plt.figure(figsize=(10, 6))
plt.plot(train_sizes, train_mean, 'o-', color='#2E86AB', label='Training Score')
plt.fill_between(train_sizes, train_mean - train_std, train_mean + train_std, alpha=0.1, color='#2E86AB')
plt.plot(train_sizes, test_mean, 'o-', color='#A23B72', label='Cross-Validation Score')
plt.fill_between(train_sizes, test_mean - test_std, test_mean + test_std, alpha=0.1, color='#A23B72')

plt.xlabel('Training Set Size')
plt.ylabel('R² Score')
plt.title('Learning Curves (Linear Regression)', fontsize=14, fontweight='bold')
plt.legend(loc='lower right')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../../day12/plots/learning_curves.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Prediction Error Analysis
errors = y_test - y_test_pred
abs_errors = np.abs(errors)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Error distribution
sns.histplot(errors, kde=True, bins=20, ax=axes[0,0], color='#C73E1D')
axes[0,0].axvline(x=0, color='black', linestyle='--', linewidth=2)
axes[0,0].set_xlabel('Prediction Error (Actual - Predicted)')
axes[0,0].set_title('Error Distribution')
axes[0,0].grid(True, alpha=0.3)

# 2. Absolute error vs Actual
axes[0,1].scatter(y_test, abs_errors, alpha=0.6, s=50, color='#F18F01')
axes[0,1].set_xlabel('Actual Score')
axes[0,1].set_ylabel('Absolute Error')
axes[0,1].set_title('Absolute Error vs Actual Score')
axes[0,1].grid(True, alpha=0.3)

# 3. Error vs Predicted
axes[1,0].scatter(y_test_pred, errors, alpha=0.6, s=50, color='#2E86AB')
axes[1,0].axhline(y=0, color='black', linestyle='--', linewidth=2)
axes[1,0].set_xlabel('Predicted Score')
axes[1,0].set_ylabel('Error')
axes[1,0].set_title('Error vs Predicted Score')
axes[1,0].grid(True, alpha=0.3)

# 4. Q-Q plot for normality
from scipy import stats
stats.probplot(errors, dist="norm", plot=axes[1,1])
axes[1,1].set_title('Q-Q Plot (Error Normality Check)')
axes[1,1].grid(True, alpha=0.3)

plt.suptitle('Prediction Error Analysis', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('../../day12/plots/error_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Model Performance Summary Table
summary_data = {
    'Metric': ['MAE', 'MSE', 'RMSE', 'R²', 'MAPE (%)', 'MedAE', 'Max Error', 'Explained Variance'],
    'Train': [
        train_metrics['Train_MAE'], train_metrics['Train_MSE'], train_metrics['Train_RMSE'],
        train_metrics['Train_R²'], train_metrics['Train_MAPE'], train_metrics['Train_MedAE'],
        train_metrics['Train_MaxError'], train_metrics['Train_ExplainedVar']
    ],
    'Test': [
        test_metrics['Test_MAE'], test_metrics['Test_MSE'], test_metrics['Test_RMSE'],
        test_metrics['Test_R²'], test_metrics['Test_MAPE'], test_metrics['Test_MedAE'],
        test_metrics['Test_MaxError'], test_metrics['Test_ExplainedVar']
    ],
    'CV Mean': [
        metadata['metrics']['cv']['mean_r2'],  # placeholder
        '-', '-',
        metadata['metrics']['cv']['mean_r2'],
        '-', '-', '-', '-'
    ]
}

summary_df = pd.DataFrame(summary_data)
print("📋 MODEL PERFORMANCE SUMMARY")
print("=" * 70)
display(summary_df.style.format({'Train': '{:.4f}', 'Test': '{:.4f}', 'CV Mean': '{:.4f}'}))

In [ ]:
# Save Evaluation Results
evaluation_results = {
    'train_metrics': train_metrics,
    'test_metrics': test_metrics,
    'cv_metrics': {
        'mean_r2': metadata['metrics']['cv']['mean_r2'],
        'std_r2': metadata['metrics']['cv']['std_r2']
    },
    'model_info': {
        'type': 'LinearRegression',
        'features': list(X.columns),
        'n_features': X.shape[1],
        'n_train': int(X_train.shape[0]),
        'n_test': int(X_test.shape[0])
    }
}

with open('../../day12/models/evaluation_results.json', 'w') as f:
    json.dump(evaluation_results, f, indent=2)

print("✅ Evaluation results saved: evaluation_results.json")

## 📝 Summary

- Comprehensive metrics calculated (MAE, MSE, RMSE, R², MAPE, MedAE, MaxError, Explained Variance)
- Learning curves show model behavior with varying training sizes
- Error analysis reveals prediction patterns
- Q-Q plot checks residual normality assumption
- Results saved for reporting

---
*Next: [06_prediction_app.ipynb](06_prediction_app.ipynb)*